# Mask Ablation Experiment — LCLD

Runs mask variants M2 (directional), M3 (derived-feature freeze), M4 (term OHE freeze), M5 (combined), M6-strict, M6-relaxed on LCLD with the neural model and CAPGD attack. Loads M0/M1 baselines from existing comparison results. Produces summary tables, feasibility audit, perturbation stats, and E1 cost-weighted analysis.

Reference: `notebooks/tabularbench_comparison.ipynb` (reused setup + utility functions).
Spec: `docs/plans/mask_ablation_experiment_plan.md`.

In [ ]:
# Cell 1: Verify GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU")

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/FraudBench"
for subdir in ["data", "results", "results/adv_examples"]:
    os.makedirs(os.path.join(DRIVE_ROOT, subdir), exist_ok=True)
print("Google Drive mounted.")

In [ ]:
# Cell 3: Clone or update repo
import os, shutil

REPO_URL = "https://github.com/iHaydenzZ/Capstone_FraudBench.git"
REPO_DIR = "/content/Capstone_FraudBench"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    os.chdir(REPO_DIR)
    !git pull
else:
    os.chdir("/content")
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone {REPO_URL} {REPO_DIR}
    os.chdir(REPO_DIR)

print(f"Working directory: {os.getcwd()}")
!git log --oneline -3

In [ ]:
# Cell 4: Install dependencies
# Colab's pre-installed numpy/scipy can conflict with project deps.
# Force a compatible set, then restart the runtime so the C extensions reload.
!pip install "numpy<2.1" "scipy>=1.14,<1.15" "scikit-learn>=1.5" -q 2>&1 | tail -5
!pip install -e . --no-deps -q 2>&1 | tail -5
!pip install "numba>=0.61" -q 2>&1 | tail -3
!pip install xgboost torch art pyyaml joblib pandas -q 2>&1 | tail -3

# --- IMPORTANT ---
# After this cell finishes, restart the runtime:
#   Runtime > Restart session  (or Ctrl+M then .)
# Then skip this cell and continue from Cell 5.
# The restart is needed because Colab caches numpy's C extensions in memory.
print("\n>>> RESTART THE RUNTIME NOW, then skip this cell and run from Cell 5. <<<")

In [ ]:
# Cell 5: Symlink datasets from Google Drive
import os

DRIVE_DATA = "/content/drive/MyDrive/FraudBench/data"
DATASETS_DIR = "/content/Capstone_FraudBench/datasets"

for dataset_dir in ["CCFD", "ieee-fraud-detection", "LCLD", "Sparkov"]:
    src = os.path.join(DRIVE_DATA, dataset_dir)
    dst = os.path.join(DATASETS_DIR, dataset_dir)
    if os.path.islink(dst):
        os.unlink(dst)
    if os.path.exists(src):
        os.symlink(src, dst)
        print(f"  Linked: {dataset_dir}/")
    else:
        print(f"  NOT FOUND: {dataset_dir}/ -- upload to {src}")

print("Dataset symlinks ready.")

In [ ]:
# Cell 6: Mask variant configuration
#
# M3/M4/M5 are defined by their IMMUTABLE raw-feature sets (extends LCLD_IMMUTABLE_RAW).
# M6-strict and M6-relaxed are defined by their MUTABLE raw-feature sets; the experiment
# loop (Cell 8) inverts them against dataset.X.columns to get the immutable set to
# pass into build_processed_mutable_mask().

from typing import Set

# --- Baseline immutable set (copied verbatim from tabularbench_comparison.ipynb Cell 6) ---
LCLD_IMMUTABLE_RAW: Set[str] = {
    # LC internal pricing/grading
    "grade", "sub_grade", "int_rate", "installment",
    "funded_amnt", "funded_amnt_inv", "initial_list_status",
    # LC verification outcomes
    "verification_status", "verification_status_joint",
    # Credit bureau data
    "delinq_2yrs", "inq_last_6mths", "inq_last_12m", "inq_fi",
    "open_acc", "open_acc_6m", "open_act_il",
    "open_il_12m", "open_il_24m", "open_rv_12m", "open_rv_24m",
    "pub_rec", "pub_rec_bankruptcies", "total_acc",
    "revol_bal", "revol_util", "il_util", "all_util",
    "tot_cur_bal", "tot_hi_cred_lim", "total_bal_il",
    "total_rev_hi_lim", "max_bal_bc",
    "pct_tl_nvr_dlq", "percent_bc_gt_75",
    "collections_12_mths_ex_med",
    "mths_since_last_delinq", "mths_since_last_il_delinq",
    "mths_since_last_major_delinq", "mths_since_last_record",
    "mths_since_rcnt_il",
    "payment_inc_ratio",
}

LCLD_MUTABLE_RAW: Set[str] = {
    "loan_amnt", "term", "purpose", "emp_length",
    "annual_inc", "annual_inc_joint", "home_ownership",
    "dti", "dti_joint", "application_type", "addr_state",
}

# --- Variant immutable sets ---
LCLD_IMMUTABLE_M3 = LCLD_IMMUTABLE_RAW | {"dti", "dti_joint"}
LCLD_IMMUTABLE_M4 = LCLD_IMMUTABLE_RAW | {"term"}
LCLD_IMMUTABLE_M5 = LCLD_IMMUTABLE_RAW | {"dti", "dti_joint", "term"}

# --- M6 attacker-capability profiles ---
MUTABLE_STRICT: Set[str] = {
    "loan_amnt", "purpose", "home_ownership", "application_type", "addr_state",
}
MUTABLE_RELAXED: Set[str] = {
    "loan_amnt", "purpose", "home_ownership", "application_type", "addr_state",
    "annual_inc", "annual_inc_joint", "emp_length",
}

# --- M2 directionality ---
# Only emp_length is increase-only among LCLD mutable features.
DIRECTION_CONSTRAINTS = {"emp_length": "increase"}

# --- E1 cost weights (raw-space, normalized units) ---
FEATURE_COSTS = {
    "loan_amnt":        1.0,
    "purpose":          0.5,
    "home_ownership":   3.0,
    "addr_state":       2.0,
    "application_type": 1.0,
    "annual_inc":       8.0,
    "annual_inc_joint": 8.0,
    "emp_length":       5.0,
    "dti":              7.0,
    "dti_joint":        7.0,
    "term":             1.0,
}

# --- Sanity checks ---
assert LCLD_IMMUTABLE_RAW.isdisjoint(LCLD_MUTABLE_RAW), "raw sets overlap"
assert set(FEATURE_COSTS.keys()) == LCLD_MUTABLE_RAW, "cost dict must cover all mutable features"
assert MUTABLE_STRICT <= LCLD_MUTABLE_RAW, "strict profile contains non-mutable feature"
assert MUTABLE_RELAXED <= LCLD_MUTABLE_RAW, "relaxed profile contains non-mutable feature"
assert LCLD_IMMUTABLE_M5 == LCLD_IMMUTABLE_M3 | LCLD_IMMUTABLE_M4, "M5 must equal M3 ∪ M4"
assert all(v in {"increase", "decrease"} for v in DIRECTION_CONSTRAINTS.values()), "unknown direction constraint"

print(f"Baseline mutable:   {len(LCLD_MUTABLE_RAW)} raw features")
print(f"M3 locks adds:      {sorted(LCLD_IMMUTABLE_M3 - LCLD_IMMUTABLE_RAW)}")
print(f"M4 locks adds:      {sorted(LCLD_IMMUTABLE_M4 - LCLD_IMMUTABLE_RAW)}")
print(f"M5 locks adds:      {sorted(LCLD_IMMUTABLE_M5 - LCLD_IMMUTABLE_RAW)}")
print(f"M6-strict mutable:  {sorted(MUTABLE_STRICT)} ({len(MUTABLE_STRICT)} features)")
print(f"M6-relaxed mutable: {sorted(MUTABLE_RELAXED)} ({len(MUTABLE_RELAXED)} features)")

In [ ]:
# Cell 7: Build processed-space mutability mask & define masked CAPGD
#
# After OneHotEncoding, categorical features expand:
#   "grade" -> "grade_A", "grade_B", ...
# All expanded columns from an immutable raw feature are also immutable.
#
# The existing project_constraints() already reverts non-numeric (OHE)
# features to original values, so those are effectively immutable.
# This mask additionally locks NUMERIC immutable features.

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from typing import Dict, Any, Optional, Set


def build_processed_mutable_mask(
    processed_feature_names: list,
    immutable_raw: Set[str],
) -> np.ndarray:
    """
    Build a boolean mask over processed (post-OHE) features.
    True = mutable (can be perturbed), False = immutable.

    OHE columns are named like "purpose_debt_consolidation" -- the prefix
    before the first underscore that matches a raw feature name determines
    mutability. For ambiguous cases (multiple underscores), we try
    progressively longer prefixes.
    """
    mask = np.ones(len(processed_feature_names), dtype=bool)  # default: mutable

    for i, col in enumerate(processed_feature_names):
        # Direct match (numeric features keep their name)
        if col in immutable_raw:
            mask[i] = False
            continue

        # OHE match: try prefixes "X" from "X_value"
        # e.g. "verification_status_Verified" -> try "verification_status"
        parts = col.split("_")
        for k in range(1, len(parts)):
            prefix = "_".join(parts[:k])
            if prefix in immutable_raw:
                mask[i] = False
                break

    return mask


def capgd_attack_masked(
    model,
    X: pd.DataFrame,
    y: pd.Series,
    schema,  # ConstraintSchema
    feature_types: Dict[str, str],
    mutable_mask: np.ndarray,
    params: Dict[str, Any] = None,
) -> pd.DataFrame:
    """
    CAPGD attack with mutability mask.
    Identical to attacks/capgd.py but zeroes gradients on immutable features.
    """
    from attacks.capgd import project_constraints

    if params is None:
        params = {}

    epsilon = params.get("epsilon", 0.1)
    steps = params.get("steps", 10)
    step_size = params.get("step_size", epsilon / 4)

    if not hasattr(model, "model") or not isinstance(model.model, nn.Module):
        print("Warning: Model is not PyTorch. Returning clean data.")
        return X

    torch_model = model.model
    device = model.device
    torch_model.eval()

    X_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1).to(device)
    feature_names = X.columns.tolist()

    # Convert mask to torch tensor on device
    mutable_t = torch.tensor(mutable_mask, dtype=torch.bool).to(device)

    # Random init within epsilon ball
    noise = torch.zeros_like(X_tensor).uniform_(-epsilon, epsilon)
    noise[:, ~mutable_t] = 0  # no perturbation on immutable features
    x_adv = X_tensor + noise
    x_adv = project_constraints(x_adv, X_tensor, schema, feature_names, feature_types)
    x_adv = x_adv.detach()
    x_adv.requires_grad = True

    use_logits = hasattr(model, "_use_logits") and model._use_logits
    criterion = nn.BCEWithLogitsLoss() if use_logits else nn.BCELoss()

    for step in range(steps):
        outputs = torch_model(x_adv)
        loss = criterion(outputs, y_tensor)

        torch_model.zero_grad()
        loss.backward()

        with torch.no_grad():
            grad = x_adv.grad
            grad[:, ~mutable_t] = 0  # KEY: zero gradients on immutable features

            x_adv = x_adv + step_size * grad.sign()

            # Project onto epsilon ball
            if epsilon > 0:
                delta = x_adv - X_tensor
                delta = torch.clamp(delta, -epsilon, epsilon)
                delta[:, ~mutable_t] = 0  # enforce zero delta on immutable
                x_adv = X_tensor + delta

            # Project onto feasibility constraints
            x_adv = project_constraints(
                x_adv, X_tensor, schema, feature_names, feature_types
            )
            x_adv.requires_grad = True

    return pd.DataFrame(
        x_adv.detach().cpu().numpy(), columns=feature_names, index=X.index
    )


print("Masked CAPGD function defined.")

def capgd_attack_masked_directional(
    model,
    X: pd.DataFrame,
    y: pd.Series,
    schema,
    feature_types: Dict[str, str],
    mutable_mask: np.ndarray,
    direction_cols: Dict[int, str],  # {processed_col_idx: "increase"|"decrease"}
    params: Dict[str, Any] = None,
) -> pd.DataFrame:
    """
    CAPGD with mask + per-column directional constraints.
    direction_cols maps processed column index to 'increase' (delta >= 0)
    or 'decrease' (delta <= 0). Applied after the epsilon-ball projection
    at every step.
    """
    from attacks.capgd import project_constraints

    if params is None:
        params = {}

    epsilon = params.get("epsilon", 0.1)
    steps = params.get("steps", 10)
    step_size = params.get("step_size", epsilon / 4)

    if not hasattr(model, "model") or not isinstance(model.model, nn.Module):
        print("Warning: Model is not PyTorch. Returning clean data.")
        return X

    torch_model = model.model
    device = model.device
    torch_model.eval()

    X_tensor = torch.tensor(X.values, dtype=torch.float32).to(device)
    y_tensor = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1).to(device)
    feature_names = X.columns.tolist()

    mutable_t = torch.tensor(mutable_mask, dtype=torch.bool).to(device)

    # Build directional clip tensors once
    inc_idx = torch.tensor(
        [i for i, d in direction_cols.items() if d == "increase"],
        dtype=torch.long, device=device,
    )
    dec_idx = torch.tensor(
        [i for i, d in direction_cols.items() if d == "decrease"],
        dtype=torch.long, device=device,
    )

    noise = torch.zeros_like(X_tensor).uniform_(-epsilon, epsilon)
    noise[:, ~mutable_t] = 0
    x_adv = X_tensor + noise
    # Apply directional clip to initial noise
    if len(inc_idx) > 0:
        x_adv[:, inc_idx] = torch.maximum(x_adv[:, inc_idx], X_tensor[:, inc_idx])
    if len(dec_idx) > 0:
        x_adv[:, dec_idx] = torch.minimum(x_adv[:, dec_idx], X_tensor[:, dec_idx])
    x_adv = project_constraints(x_adv, X_tensor, schema, feature_names, feature_types)
    x_adv = x_adv.detach()
    x_adv.requires_grad = True

    use_logits = hasattr(model, "_use_logits") and model._use_logits
    criterion = nn.BCEWithLogitsLoss() if use_logits else nn.BCELoss()

    for step in range(steps):
        outputs = torch_model(x_adv)
        loss = criterion(outputs, y_tensor)

        torch_model.zero_grad()
        loss.backward()

        with torch.no_grad():
            grad = x_adv.grad
            grad[:, ~mutable_t] = 0

            x_adv = x_adv + step_size * grad.sign()

            if epsilon > 0:
                delta = x_adv - X_tensor
                delta = torch.clamp(delta, -epsilon, epsilon)
                delta[:, ~mutable_t] = 0
                x_adv = X_tensor + delta

            # Directional clip (processed space; StandardScaler preserves direction)
            if len(inc_idx) > 0:
                x_adv[:, inc_idx] = torch.maximum(x_adv[:, inc_idx], X_tensor[:, inc_idx])
            if len(dec_idx) > 0:
                x_adv[:, dec_idx] = torch.minimum(x_adv[:, dec_idx], X_tensor[:, dec_idx])

            x_adv = project_constraints(
                x_adv, X_tensor, schema, feature_names, feature_types
            )
            x_adv.requires_grad = True

    return pd.DataFrame(
        x_adv.detach().cpu().numpy(), columns=feature_names, index=X.index
    )


print("Directional CAPGD function defined.")

def resolve_direction_indices(
    processed_feature_names: list,
    direction_raw: Dict[str, str],
) -> Dict[int, str]:
    """Map raw feature directional config to processed column indices.

    For numeric raw features the processed name equals the raw name.
    For OHE-expanded categoricals, matches by prefix. M2 currently only
    uses numeric features (emp_length), so prefix matching is defensive.
    """
    out: Dict[int, str] = {}
    for i, col in enumerate(processed_feature_names):
        if col in direction_raw:
            out[i] = direction_raw[col]
            continue
        parts = col.split("_")
        for k in range(1, len(parts)):
            prefix = "_".join(parts[:k])
            if prefix in direction_raw:
                out[i] = direction_raw[prefix]
                break
    # TODO(debt): prefix matching is greedy-shortest-prefix; if a column shares a
    # prefix with a raw feature name (e.g. 'emp_length_years' ~ 'emp_length'), it
    # will inherit that feature's direction. Safe for current LCLD schema.
    return out


print("Direction index resolver defined.")

In [ ]:
# Cell 8: Main experiment loop — train once per seed, run all 6 new variants
import time
import os
import numpy as np
import pandas as pd

from datasets.loader import load_dataset
from datasets.splitter import split_dataset
from preprocessing.processor import DataPreprocessor, get_preprocessor_path
from constraints.schema import ConstraintSchema
from models.neural import NeuralModel
from evaluation.metrics import compute_metrics

SEEDS = [42, 123, 456]
EPSILON = 0.1
SAMPLE_FRAC = 0.1
ATTACK_PARAMS = {"epsilon": EPSILON, "steps": 10, "step_size": EPSILON / 4}
MODEL_PARAMS = {"epochs": 20, "hidden_dim": 128, "batch_size": 256, "lr": 0.001}

ADV_SAVE_DIR = "results/adv_examples/mask_ablation"
os.makedirs(ADV_SAVE_DIR, exist_ok=True)

# {variant_name: (attack_fn_kind, immutable_set_or_None, extra)}
# attack_fn_kind in {"masked", "directional"}.
# For M6 the immutable set is computed per-seed from dataset.X.columns.
VARIANTS = {
    "M2":         ("directional", LCLD_IMMUTABLE_RAW, {"direction": DIRECTION_CONSTRAINTS}),
    "M3":         ("masked",      LCLD_IMMUTABLE_M3,  {}),
    "M4":         ("masked",      LCLD_IMMUTABLE_M4,  {}),
    "M5":         ("masked",      LCLD_IMMUTABLE_M5,  {}),
    "M6strict":   ("masked",      "from_profile",      {"profile": MUTABLE_STRICT}),
    "M6relaxed":  ("masked",      "from_profile",      {"profile": MUTABLE_RELAXED}),
}

rows = []  # per-(variant, seed) row for mask_ablation_results.csv

for seed in SEEDS:
    print(f"\n{'='*60}\n  SEED = {seed}\n{'='*60}")

    # --- Load, split, preprocess (reuse saved preprocessor) ---
    dataset = load_dataset("lcld", config={"sample_frac": SAMPLE_FRAC})
    X_train, X_val, X_test, y_train, y_val, y_test = split_dataset(
        dataset, test_size=0.2, val_size=0.2, random_state=seed,
    )
    preprocessor_path = get_preprocessor_path("lcld", seed, len(dataset.X))
    assert os.path.exists(preprocessor_path), (
        f"Expected preprocessor at {preprocessor_path} (should have been saved by "
        f"tabularbench_comparison.ipynb). Run that notebook first or sync from Drive."
    )
    preprocessor = DataPreprocessor.load(preprocessor_path)
    X_train_p = preprocessor.transform(X_train)
    X_test_p = preprocessor.transform(X_test)
    processed_feature_types = {c: "numeric" for c in X_train_p.columns}
    processed_schema = ConstraintSchema.from_data(X_train_p, processed_feature_types)
    print(f"  Data: train={len(X_train)}, test={len(X_test)}, processed={X_test_p.shape[1]}")

    # --- Train once ---
    model = NeuralModel(MODEL_PARAMS)
    t0 = time.time()
    model.fit(X_train_p, y_train)
    print(f"  Trained in {time.time()-t0:.1f}s")

    clean_probs = model.predict_proba(X_test_p)
    clean_metrics = compute_metrics(y_test, clean_probs)
    print(f"  Clean  -- PR-AUC: {clean_metrics['pr_auc']:.4f}, Acc: {clean_metrics['accuracy']:.4f}")

    # Raw feature universe for M6 immutable-set construction
    raw_all = set(dataset.X.columns)

    # --- Loop over variants ---
    for vname, (kind, imm, extra) in VARIANTS.items():
        if imm == "from_profile":
            immutable_raw = raw_all - extra["profile"]
        else:
            immutable_raw = imm

        mutable_mask = build_processed_mutable_mask(
            X_test_p.columns.tolist(), immutable_raw
        )
        n_mut = int(mutable_mask.sum())
        n_imm = int((~mutable_mask).sum())

        t0 = time.time()
        if kind == "directional":
            dir_idx = resolve_direction_indices(
                X_test_p.columns.tolist(), extra["direction"]
            )
            X_adv = capgd_attack_masked_directional(
                model, X_test_p, y_test, processed_schema, processed_feature_types,
                mutable_mask, dir_idx, params=ATTACK_PARAMS,
            )
        else:
            X_adv = capgd_attack_masked(
                model, X_test_p, y_test, processed_schema, processed_feature_types,
                mutable_mask, params=ATTACK_PARAMS,
            )
        dt = time.time() - t0

        robust_probs = model.predict_proba(X_adv)
        robust = compute_metrics(y_test, robust_probs)
        print(
            f"  {vname:10s} mut={n_mut:3d} imm={n_imm:3d} "
            f"-- Robust PR-AUC: {robust['pr_auc']:.4f}, Acc: {robust['accuracy']:.4f} ({dt:.1f}s)"
        )

        parquet_path = os.path.join(ADV_SAVE_DIR, f"lcld_neural_{vname}_seed{seed}.parquet")
        X_adv.to_parquet(parquet_path)

        rows.append({
            "variant": vname, "seed": seed,
            "n_mutable": n_mut, "n_immutable": n_imm,
            "clean_pr_auc":  clean_metrics["pr_auc"],
            "clean_accuracy": clean_metrics["accuracy"],
            "clean_recall":   clean_metrics.get("recall", np.nan),
            "clean_f1":       clean_metrics.get("f1", np.nan),
            "robust_pr_auc":  robust["pr_auc"],
            "robust_accuracy": robust["accuracy"],
            "robust_recall":   robust.get("recall", np.nan),
            "robust_f1":       robust.get("f1", np.nan),
            "attack_time_s": dt,
        })

results_df = pd.DataFrame(rows)
results_df.to_csv(os.path.join(ADV_SAVE_DIR, "mask_ablation_results.csv"), index=False)
print(f"\nSaved per-seed results to {ADV_SAVE_DIR}/mask_ablation_results.csv")
print(results_df)

In [ ]:
# Cell 9: Load M0 (no-mask) and M1 (binary-mask) baselines into the same results frame
import shutil

BASELINE_SRC = "/content/drive/MyDrive/FraudBench/results/adv_examples"
BASELINE_LOCAL = "results/adv_examples"  # parquets already produced by reference notebook

# Copy baseline parquets locally if missing (they may only exist on Drive)
os.makedirs(BASELINE_LOCAL, exist_ok=True)
for seed in SEEDS:
    for variant_label, baseline_file in [
        ("M0", f"lcld_neural_unmasked_seed{seed}.parquet"),
        ("M1", f"lcld_neural_masked_seed{seed}.parquet"),
    ]:
        src = os.path.join(BASELINE_SRC, baseline_file)
        dst = os.path.join(BASELINE_LOCAL, baseline_file)
        if not os.path.exists(dst) and os.path.exists(src):
            shutil.copy(src, dst)

# Load comparison CSVs to get the robust metrics the baseline notebook recorded.
m0_csv = os.path.join(BASELINE_SRC, "comparison_unmasked.csv")
m1_csv = os.path.join(BASELINE_SRC, "comparison_masked.csv")
assert os.path.exists(m0_csv) and os.path.exists(m1_csv), (
    "Baseline comparison CSVs not found on Drive. Run tabularbench_comparison.ipynb first."
)
m0 = pd.read_csv(m0_csv)
m1 = pd.read_csv(m1_csv)

def _canon(df: pd.DataFrame, variant: str) -> pd.DataFrame:
    """Normalize baseline rows to the same schema as results_df."""
    out = pd.DataFrame({
        "variant": variant,
        "seed": df["seed"].astype(int),
        "n_mutable":   df.get("n_mutable",   np.nan),
        "n_immutable": df.get("n_immutable", np.nan),
        "clean_pr_auc":   df.get("clean_pr_auc",   np.nan),
        "clean_accuracy": df.get("clean_accuracy", np.nan),
        "clean_recall":   df.get("clean_recall",   np.nan),
        "clean_f1":       df.get("clean_f1",       np.nan),
        "robust_pr_auc":   df["robust_pr_auc"],
        "robust_accuracy": df["robust_accuracy"],
        "robust_recall":   df.get("robust_recall", np.nan),
        "robust_f1":       df.get("robust_f1",     np.nan),
        "attack_time_s":   df.get("attack_time_s", np.nan),
    })
    return out

baseline_rows = pd.concat([_canon(m0, "M0"), _canon(m1, "M1")], ignore_index=True)
all_results = pd.concat([baseline_rows, results_df], ignore_index=True)
all_results.to_csv(os.path.join(ADV_SAVE_DIR, "mask_ablation_results.csv"), index=False)
print(f"Combined results ({len(all_results)} rows = 8 variants × 3 seeds):")
print(all_results)

In [ ]:
def inverse_transform_numeric(X_proc, num_feature_names, scaler):
    """
    Inverse-transform only the numeric columns from processed space
    back to raw space using the fitted StandardScaler.
    """
    sanitize = lambda c: re.sub(r"[\[\]<>]", "_", c)
    sanitized_num = [sanitize(c) for c in num_feature_names]

    proc_cols = X_proc.columns.tolist()
    matched = [(raw, san) for raw, san in zip(num_feature_names, sanitized_num) if san in proc_cols]

    raw_names = [m[0] for m in matched]
    san_names = [m[1] for m in matched]
    idx_in_scaler = [num_feature_names.index(r) for r in raw_names]

    X_scaled = X_proc[san_names].values
    means = scaler.mean_[idx_in_scaler]
    scales = scaler.scale_[idx_in_scaler]
    X_raw_vals = X_scaled * scales + means

    return pd.DataFrame(X_raw_vals, columns=raw_names, index=X_proc.index)

def _to_float(series):
    """Coerce a Series to float, stripping non-numeric chars."""
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(float)
    return pd.to_numeric(series.astype(str).str.replace(r"[^\d.\-]", "", regex=True), errors="coerce")

def reconstruct_term_from_ohe(X_proc):
    """Reconstruct raw term value (36 or 60) from OHE columns in processed space.

    OHE columns are named like 'term_ 36 months'. We assign term based on
    which column has the highest activation. This is approximate (OHE values
    may be continuous after perturbation) but lets us evaluate g1.
    """
    proc_cols = X_proc.columns.tolist()
    term_cols = [c for c in proc_cols if c.startswith("term_")]
    if len(term_cols) == 0:
        return None

    term_vals = {}
    for col in term_cols:
        val = pd.to_numeric(
            col.replace("term_", "").replace("months", "").strip(),
            errors="coerce",
        )
        if not np.isnan(val):
            term_vals[col] = val

    if not term_vals:
        return None

    term_df = X_proc[list(term_vals.keys())]
    best_col = term_df.idxmax(axis=1)
    return best_col.map(term_vals)

def check_g4_processed(X_proc):
    """g4 in processed space: check that term OHE is still valid one-hot.

    A valid one-hot = exactly one column close to 1, sum close to 1.
    If CAPGD pushed both to continuous values, the encoding is broken.
    """
    proc_cols = X_proc.columns.tolist()
    term_cols = [c for c in proc_cols if c.startswith("term_")]
    if len(term_cols) < 2:
        return None

    term_arr = X_proc[term_cols].values
    row_max = term_arr.max(axis=1)
    row_sum = term_arr.sum(axis=1)

    tol = 0.01
    valid = (np.abs(row_max - 1.0) < tol) & (np.abs(row_sum - 1.0) < tol)
    return pd.Series(valid, index=X_proc.index)

def check_g1_installment(df, tol=0.1):
    """g1: installment = loan_amnt * (r*(1+r)^t) / ((1+r)^t - 1)."""
    needed = ["loan_amnt", "int_rate", "installment", "term"]
    if not all(c in df.columns for c in needed):
        return None
    loan = _to_float(df["loan_amnt"])
    rate = _to_float(df["int_rate"])
    inst = _to_float(df["installment"])
    t = _to_float(df["term"])
    r = rate / 1200.0
    expected = loan * (r * (1 + r) ** t) / ((1 + r) ** t - 1)
    diff = (inst - expected).abs()
    return diff <= tol

def check_g2_open_total(df):
    """g2: open_acc <= total_acc."""
    needed = ["open_acc", "total_acc"]
    if not all(c in df.columns for c in needed):
        return None
    return _to_float(df["open_acc"]) <= _to_float(df["total_acc"]) + TOLERANCE

def check_g3_bankruptcies(df):
    """g3: pub_rec_bankruptcies <= pub_rec."""
    needed = ["pub_rec_bankruptcies", "pub_rec"]
    if not all(c in df.columns for c in needed):
        return None
    return _to_float(df["pub_rec_bankruptcies"]) <= _to_float(df["pub_rec"]) + TOLERANCE

def check_g4_term(df):
    """g4: term must be 36 or 60 (raw-space check)."""
    if "term" not in df.columns:
        return None
    t = _to_float(df["term"])
    return t.round().isin([36, 60])

def check_g5_ratio_loan_inc(df, tol=0.01):
    """g5: ratio_loan_amnt_annual_inc == loan_amnt / annual_inc."""
    needed = ["loan_amnt", "annual_inc", "ratio_loan_amnt_annual_inc"]
    if not all(c in df.columns for c in needed):
        return None
    loan = _to_float(df["loan_amnt"])
    inc = _to_float(df["annual_inc"]).replace(0, np.nan)
    expected = loan / inc
    return (_to_float(df["ratio_loan_amnt_annual_inc"]) - expected).abs() <= tol

def check_g6_ratio_open_total(df, tol=0.01):
    """g6: ratio_open_acc_total_acc == open_acc / total_acc."""
    needed = ["open_acc", "total_acc", "ratio_open_acc_total_acc"]
    if not all(c in df.columns for c in needed):
        return None
    o = _to_float(df["open_acc"])
    t = _to_float(df["total_acc"]).replace(0, np.nan)
    expected = o / t
    return (_to_float(df["ratio_open_acc_total_acc"]) - expected).abs() <= tol

def compute_aggregate_feasibility(df, X_proc=None):
    """Check g1 (tol=G1_TOL), g2, g3, g4 simultaneously.

    For adversarial data, pass X_proc to use processed-space OHE check
    for g4 instead of argmax-reconstructed term (which always passes
    even when OHE encoding is broken).
    """
    checks = []

    g1 = check_g1_installment(df, tol=G1_TOL)
    if g1 is not None:
        checks.append(g1.fillna(True))

    g2 = check_g2_open_total(df)
    if g2 is not None:
        checks.append(g2.fillna(True))

    g3 = check_g3_bankruptcies(df)
    if g3 is not None:
        checks.append(g3.fillna(True))

    # g4: use processed-space OHE validity when available (adversarial),
    # fall back to raw-space check (clean data)
    if X_proc is not None:
        g4 = check_g4_processed(X_proc)
    else:
        g4 = check_g4_term(df)
    if g4 is not None:
        checks.append(g4.fillna(True))

    if not checks:
        return None, None, 0

    all_pass = checks[0]
    for c in checks[1:]:
        all_pass = all_pass & c

    n_constraints = len(checks)
    return all_pass, float(all_pass.mean()), n_constraints

print("Feasibility audit functions loaded.")